# OADEV at the conventional frequency-stability taus

Point values of the overlapping Allan deviation at **tau = 0.1, 1 and 10 s**, the
convention used for reporting laser frequency stability. No region averaging:
each number is `oadev` at that single tau, not a mean over a tau band.

`allantools.oadev` is called directly, so the quoted error is the one allantools
returns (`dev / sqrt(n)`), matching how errors are reported in the results chapter.
`ltatools` is used only to read the `.lta` files and to find the stable segment of
the pre-modification traces.

Two tables:

1. **Single trace** -- the same 30 s segments plotted as scenarios A-D in `WVM_thesis.ipynb`.
2. **Run to run** -- every consecutive 30 s window of the repeat traces, reported as
   mean +/- std across windows. Only C and D have repeats.

Caveat that governs the whole notebook: the segments are 30 s long, so tau = 10 s is
sampled about three times. The allantools error is the precision of the estimator on
one trace and does not carry that; the run-to-run spread in table 2 does.

Output is for the **appendix only**.

In [1]:
import allantools as at
import ltatools as lta          # .lta reader and stable-segment finder only
import numpy as np
import pandas as pd

TAUS   = [0.1, 1.0, 10.0]       # s, the reporting convention
WINDOW = 30                     # s, run-to-run window length
THz_Hz = 1e12

In [2]:
def segment(df, t0, t1):
    """Slice [t0, t1) seconds and re-zero the clock, as in WVM_thesis.ipynb."""
    s = df[(df['time_s'] > t0) & (df['time_s'] < t1)].reset_index(drop=True)
    s['time_s'] = s['time_s'] - s['time_s'].iloc[0]
    return s


def first_stable(path):
    """Pre-modification traces are cleaned and cut to their first stable segment."""
    return next(iter(lta.find_stable_segments(lta.load_lta_file(path, cleanup=True), n=1)))


def oadev_at(df, taus=TAUS):
    """allantools.oadev on the absolute frequency, in Hz. Returns (tau, dev, err, n).

    The wavemeter timestamps are evenly spaced to far better than one gate time,
    so the rate is taken as the reciprocal of the median sample interval.
    """
    rate = 1.0 / np.median(np.diff(df['time_s']))
    y = df['frequency_THz'].to_numpy() * THz_Hz
    return at.oadev(y, rate=rate, data_type='freq', taus=taus)


# Scenario definitions copied from WVM_thesis.ipynb so the two agree by construction.
SCENARIOS = {
    'A': segment(first_stable('data/24.06.2026, 14.08,  268,0946227 THz.lta'), 32, 62),
    'B': segment(first_stable('data/24.06.2026, 14.33,  268,0950613 THz.lta'),  5, 35),
    'C': segment(lta.load_lta_file('data/02.07.2026, 19.15,  268,0959664 THz.lta'), 30, 60),
    'D': segment(lta.load_lta_file('data/06.07.2026, 15.12,  268,0962819 THz.lta'),  0, 30),
}

for name, df in SCENARIOS.items():
    gate = np.median(np.diff(df['time_s'])) * 1e3
    print(f'{name}: N={len(df):5d}  span={df["time_s"].max():5.1f} s  gate={gate:5.1f} ms')

A: N= 2919  span= 30.0 s  gate= 10.0 ms
B: N= 2972  span= 30.0 s  gate= 10.0 ms
C: N= 1916  span= 30.0 s  gate= 16.0 ms
D: N= 2410  span= 30.0 s  gate= 12.0 ms


## Table 1 -- single trace

The error column is `allantools`' own: `dev / sqrt(n)`. It is the precision of the
estimator on this one trace and does not account for the noise type, for the
correlation that overlapping samples introduce, or for measurement-to-measurement
scatter. Table 2 is the place to look for the latter.

In [3]:
rows = []
for name, df in SCENARIOS.items():
    tau, dev, err, n = oadev_at(df)
    for t_req, t_used, d, e, ni in zip(TAUS, tau, dev, err, n):
        rows.append({'scenario': name, 'tau_req_s': t_req, 'tau_used_s': t_used,
                     'sigma_kHz': d / 1e3, 'err_kHz': e / 1e3, 'n_pairs': int(ni)})

single = pd.DataFrame(rows)
single

,scenario,tau_req_s,tau_used_s,sigma_kHz,err_kHz,n_pairs
0,A,0.1,0.100,152.217581,2.826610,2900
1,A,1.0,1.000,1178.057185,22.588215,2720
2,A,10.0,10.000,3408.322825,112.369076,920
3,B,0.1,0.100,672.751480,12.380065,2953
4,B,1.0,1.000,6694.304842,127.124878,2773
5,B,10.0,10.000,66217.807184,2122.845438,973
6,C,0.1,0.096,104.986247,2.405387,1905
7,C,1.0,1.008,775.286583,18.319536,1791
8,C,10.0,10.000,3511.315951,135.958697,667
9,D,0.1,0.096,76.058883,1.554165,2395


## Table 2 -- run to run

Every consecutive 30 s window of the repeat traces, then mean +/- std across windows.
A and B have no repeat measurements, so they cannot appear here.

In [4]:
REPEATS = {
    'C': ['data/02.07.2026, 18.13,  268,0962595 THz.lta',
          'data/02.07.2026, 19.15,  268,0959664 THz.lta'],
    'D': ['data/06.07.2026, 15.12,  268,0962819 THz.lta',
          'data/06.07.2026, 14.55,  268,0962734 THz.lta',
          'data/06.07.2026, 14.45,  268,0962640 THz.lta'],
}

per_window = []
for name, paths in REPEATS.items():
    for path in paths:
        df = lta.load_lta_file(path)
        for i in range(int(df['time_s'].max()) // WINDOW):
            w = segment(df, i * WINDOW, (i + 1) * WINDOW)
            if len(w) < 50:
                continue
            _, dev, _, _ = oadev_at(w)
            for t_req, d in zip(TAUS, dev):
                per_window.append({'scenario': name, 'file': path.split('/')[-1],
                                   'window': i, 'tau_req_s': t_req, 'sigma_kHz': d / 1e3})

per_window = pd.DataFrame(per_window)

runs = (per_window.groupby(['scenario', 'tau_req_s'])['sigma_kHz']
        .agg(mean='mean', std=lambda s: s.std(ddof=1), min='min', max='max', n_windows='size')
        .reset_index())
runs

,scenario,tau_req_s,mean,std,min,max,n_windows
0,C,0.1,114.841026,24.680542,87.809568,168.525128,11
1,C,1.0,867.030193,275.173333,519.005505,1424.030939,11
2,C,10.0,4405.576084,2358.264693,1080.552523,9276.170969,11
3,D,0.1,85.619214,5.569912,76.058883,93.548897,9
4,D,1.0,176.787403,38.520191,122.827017,228.724772,9
5,D,10.0,724.722455,462.973533,331.309812,1658.550897,9


## LaTeX

Printed, not written to disk. Paste into the appendix.

In [5]:
import math


def si(value, err):
    """siunitx concise notation, error rounded to 2 significant figures.

    Matches the convention already used in the results chapter (876(4) kHz),
    where the value is quoted to the same decimal place as the error.
    kHz below 1 MHz, MHz above.
    """
    unit, scale = ('\\mega\\hertz', 1000.0) if value >= 1000 else ('\\kilo\\hertz', 1.0)
    v, e = value / scale, err / scale
    d = -int(math.floor(math.log10(abs(e)))) + 1      # decimals for 2 sig figs on err
    v, e = round(v, d), round(e, d)
    p = max(d, 0)
    return f'\\SI{{{v:.{p}f} \\pm {e:.{p}f}}}{{{unit}}}'


print('% ---- single trace, allantools oadev error ----')
for name in SCENARIOS:
    sub = single[single['scenario'] == name]
    print(f'\\textbf{{{name}}} & '
          + ' & '.join(si(r['sigma_kHz'], r['err_kHz']) for _, r in sub.iterrows())
          + ' \\\\')

print()
print('% ---- run to run, mean +/- std over 30 s windows ----')
for name in REPEATS:
    sub = runs[runs['scenario'] == name]
    print(f'\\textbf{{{name}}} & '
          + ' & '.join(si(r['mean'], r['std']) for _, r in sub.iterrows())
          + f' & {int(sub["n_windows"].iloc[0])} \\\\')


% ---- single trace, allantools oadev error ----
\textbf{A} & \SI{152.2 \pm 2.8}{\kilo\hertz} & \SI{1.178 \pm 0.023}{\mega\hertz} & \SI{3.41 \pm 0.11}{\mega\hertz} \\
\textbf{B} & \SI{673 \pm 12}{\kilo\hertz} & \SI{6.69 \pm 0.13}{\mega\hertz} & \SI{66.2 \pm 2.1}{\mega\hertz} \\
\textbf{C} & \SI{105.0 \pm 2.4}{\kilo\hertz} & \SI{775 \pm 18}{\kilo\hertz} & \SI{3.51 \pm 0.14}{\mega\hertz} \\
\textbf{D} & \SI{76.1 \pm 1.6}{\kilo\hertz} & \SI{228.7 \pm 4.8}{\kilo\hertz} & \SI{1.659 \pm 0.061}{\mega\hertz} \\

% ---- run to run, mean +/- std over 30 s windows ----
\textbf{C} & \SI{115 \pm 25}{\kilo\hertz} & \SI{870 \pm 280}{\kilo\hertz} & \SI{4.4 \pm 2.4}{\mega\hertz} & 11 \\
\textbf{D} & \SI{85.6 \pm 5.6}{\kilo\hertz} & \SI{177 \pm 39}{\kilo\hertz} & \SI{720 \pm 460}{\kilo\hertz} & 9 \\
